In [5]:
# we need it in all codes
import numpy as np 
import math

import scipy.sparse
from scipy.sparse import csr_matrix
# see functions here
# https://numpy.org/doc/stable/reference/generated/numpy.exp.html
# https://numpy.org/doc/stable/reference/routines.array-manipulation.html
#

# basic vector and matrix operations
# general principle: we do not have to write functions for everything
# moreover, built-in vectorized operations are rather fast


# SPARSE MATRICES

In [18]:
M = np.array([[1, 0, 0, 0, 0, 1], [0, 3, 0, 0, 0, 1], [0, 0, 0, 2, 0, 0], [0, 1, 0, 0, 4, 0], [4, 0, 0, 1, 0, 0],[0, 0, 0, 3, 0, 0]])
print('Let us print M in dense form:')
print(f"M={M}")
# convert to sparse matrix
S = csr_matrix(M)
print("Now M is a sparse matrix, let's call it S:")
print(f"S={S}")
# reconstruct dense matrix
print("We can recover M using .todense() method, lets call this B and see if we get back M:")
B = S.todense() 
print(f"B = {B}")
# check whether S is really sparse
print("Let's check if S is sparse:")
print(scipy.sparse.issparse(S))
S2 = S@S
# check whether multiplication preserves sparsity:
print("Is S^2 sparse?:")
print(scipy.sparse.issparse(S2))
# be very careful: a simple identical operation can harm this structure
print("What about multiplying a sparse matrix with a non sparse matrix?")
S22 = np.eye(6)@S2 #scipy.sparse.eye for a sparse id matrix.
scipy.sparse.issparse(S22)


Let us print M in dense form:
M=[[1 0 0 0 0 1]
 [0 3 0 0 0 1]
 [0 0 0 2 0 0]
 [0 1 0 0 4 0]
 [4 0 0 1 0 0]
 [0 0 0 3 0 0]]
Now M is a sparse matrix, let's call it S:
S=<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 10 stored elements and shape (6, 6)>
  Coords	Values
  (0, 0)	1
  (0, 5)	1
  (1, 1)	3
  (1, 5)	1
  (2, 3)	2
  (3, 1)	1
  (3, 4)	4
  (4, 0)	4
  (4, 3)	1
  (5, 3)	3
We can recover M using .todense() method, lets call this B and see if we get back M:
B = [[1 0 0 0 0 1]
 [0 3 0 0 0 1]
 [0 0 0 2 0 0]
 [0 1 0 0 4 0]
 [4 0 0 1 0 0]
 [0 0 0 3 0 0]]
Let's check if S is sparse:
True
Is S^2 sparse?:
True
What about multiplying a sparse matrix with a non sparse matrix?


False

In [7]:
# Measure how much faster is it to compute matrices in this form
# take the previous one and compute manually the 10th power of it.

from timeit import default_timer as timer

# Try to multiply sparse matrices first.
start = timer()
S10 = S@S@S@S@S@S@S@S@S@S
end = timer()
print(f'The time it takes to multipy a sparse matrix 10 times is {end-start}')


# Also, try this using the dense form.
start = timer()
B10 = B@B@B@B@B@B@B@B@B@B
end = timer()
print(f'The time it takes to multipy its dense form 10 times  is {end-start}')

print("Let's see how different are these outcomes:")
print(B10-S10)

# What happened?
# This is rather strange.
# For these small matrices we should not play with this. It is not worth it to restructure a matrix when it's small.

The time it takes to multipy a sparse matrix 10 times is 0.00038920799852348864
The time it takes to multipy its dense form 10 times  is 0.00033824998536147177
Let's see how different are these outcomes:
[[0 0 0 0 0 0]
 [0 0 0 0 0 0]
 [0 0 0 0 0 0]
 [0 0 0 0 0 0]
 [0 0 0 0 0 0]
 [0 0 0 0 0 0]]


In [8]:
e = np.ones(5)
np.array([e,2*e, -e])

array([[ 1.,  1.,  1.,  1.,  1.],
       [ 2.,  2.,  2.,  2.,  2.],
       [-1., -1., -1., -1., -1.]])

# CONSTRUCTING SPARSE DIAGONAL MATRICES

In [9]:
from scipy.sparse import dia_matrix
n = 1000
ex = np.ones(n)
#Lets build a tridiagonal sparse matrix
# 
# # We first create an array consisting of the diagonals. Store them row-wise
data = np.array([ex, 2 * ex, ex])
# We also need an array that tells us n which diagonals should the rows be inserted
# (related to the main diagonal. That is, main diag =0, above =1, below -1,...). 
offsets = np.array([-1, 0, 1])
# Then the matrix is collected. We will use dia_matrix:
#dia_matrix((data, offsets), shape=(number_of_rows, number_of_columns))
#This gives already a sparse matrix, to see its dense form, we use the method .toarray()

trd = dia_matrix((data, offsets), shape=(n, n)).toarray() 
# Be careful; this should still be converted to a sparse matrix.
S_1000 = csr_matrix(trd)
print(f" Is this matrix converted succesfully to a sparse one: {scipy.sparse.issparse(S_1000)}")

#Testing multiplication times:
start = timer()
S_1000_10 = S_1000@S_1000@S_1000@S_1000@S_1000@S_1000@S_1000@S_1000@S_1000@S_1000
end = timer()
mult_sparse_time = end-start
print(f"The time it takes to multiply this matrix 9 times is: {mult_sparse_time}")

#Comparing to a dense matrix:
start = timer()
trd_10 = trd@trd@trd@trd@trd@trd@trd@trd@trd@trd
end = timer()
mult_dense_time = end-start
print(f"Now, this is the time it takes to multiplicate the same matrix in dense form 9 times:{mult_dense_time}")

# Now anyone can see the difference.
print(f"Multiplying sparse matrices is {mult_dense_time/mult_sparse_time} faster")


 Is this matrix converted succesfully to a sparse one: True
The time it takes to multiply this matrix 9 times is: 0.0014483329723589122
Now, this is the time it takes to multiplicate the same matrix in dense form 9 times:0.03856066701700911
Multiplying sparse matrices is 26.624172585261952 faster


# SOLVING LINEAR SYSTEMS

In [10]:
#Let's solve first a lower triagonal system!
#We will build our first linear solver :)
n=500
A = np.zeros((n,n))
#row i, we rewrite from the first column until the ith so 0: i+1 (end is omitted thats why)
for i  in np.arange(n):
    A [i, 0:i+1] = np.arange(1, i+2) # write numbers from 1 up to i+1  
print(A)

#Let's take a random vector for the RHS
b_random = np.random.rand(n,)

#Solving the system using forward substitution:
#We will build our first linear solver :)
sol = np.zeros(n,)
sol[0] = b_random[0] / A[0,0]
for i in np.arange(1,n):
    sol[i] = (1 / A[i,i]) * (b_random[i]- A[i, 0 : i]  @ sol[0:i])

[[  1.   0.   0. ...   0.   0.   0.]
 [  1.   2.   0. ...   0.   0.   0.]
 [  1.   2.   3. ...   0.   0.   0.]
 ...
 [  1.   2.   3. ... 498.   0.   0.]
 [  1.   2.   3. ... 498. 499.   0.]
 [  1.   2.   3. ... 498. 499. 500.]]


In [11]:
np.allclose(A@ sol ,b_random)

True

## LU Decomposition

In [12]:
# Solve the linear system using "np.linalg.solve"
#  2x + 3y - 4z = 12
# -4x - y  + 5z = -11
# -6x + 2y - z  = -1

# It is based on the so-called LU decomposition, where 
# A = L@U  a product of lower and upper triangle matrix.
# Then Ax = b equivalent to LU x = b. Set xx = U x to obtain
# the system L xx = b. Solve this to obtain xx, then solve 
# the system U x = xx to finally obtain the solution x.

#linalg. solve is a bit more sofisticated than that since it also
#permutes the rows of A for better factorization. 
A = np.array([[2, 3, -4],[-4, -1, 5],[-6, 2, -1]])
b = np.array([12, -11, -1])
x = np.linalg.solve(A, b)

print(f"The solution obtained from LU factorization is {x}")


The solution obtained from LU factorization is [ 1.  2. -1.]


In [13]:

# What happens if it cannot be solved?
# 
#  2x + 3y - 4z = 12
#  4x + 6y - 8z = 20
# -6x + 2y - z  = -1

# Here, if it could be solved, we should have 
#  20 = 2 * 12.

A2 = np.array([[2, 3, -4],[4, 6, -8],[-6, 2, -1]])
b2 = np.array([12, 20, -1])
#x2 = np.linalg.solve(A2, b2)

# 


# solve the original one using matrix inverses
#Ainv = np.linalg.inv(A)
#print(Ainv)
#print (Ainv@b)

#  of course, this gives the same

# A COMPETITION OF FIVE METHODS

In [14]:
# Try to solve a linear system applying three methods.
# 1. Built-in solver using sparse structure
# 2. Built-in solver without using sparse structure
# 3. Using inverse.
# 4. Appyling Jacobi iteration and using sparse structure
# 5. Appyling Jacobi iteration and using sparse structure


from scipy.sparse import dia_matrix
from scipy.sparse import csc_matrix
from scipy.sparse.linalg import spsolve
n = 10000
ex = np.ones(n)
# This will be a tridiagonal matrix
# We first create an array consisting of the diagonals.
data = np.array([ex, -4 * ex, ex])
# We also say, in which diagonals should the rows be inserted
# (related to the main diagonal).
offsets = np.array([-1, 0, 1])
# Then the matrix is collected.
trd = dia_matrix((data, offsets), shape=(n, n))
# Be careful; if we use np.copy(trd) we will obtain an np.array that contains inside the sparse matrix. 
#Let's use trd's own copy method instead.
S_10000 = trd.copy()
S_10000 = csr_matrix(trd)
S_dense = trd.toarray() #let's keep a dense copy too


print(scipy.sparse.issparse(S_10000))

#Since trd is sparse we cant access their entries that easily
#print(trd[4,4]) wont work.


True


In [15]:
S_dense

array([[-4.,  1.,  0., ...,  0.,  0.,  0.],
       [ 1., -4.,  1., ...,  0.,  0.,  0.],
       [ 0.,  1., -4., ...,  0.,  0.,  0.],
       ...,
       [ 0.,  0.,  0., ..., -4.,  1.,  0.],
       [ 0.,  0.,  0., ...,  1., -4.,  1.],
       [ 0.,  0.,  0., ...,  0.,  1., -4.]], shape=(10000, 10000))

In [ ]:

# Construct a test problem
x_known = np.random.rand(10000)
b = S_dense@x_known

# Use conventional solver and non-sparse form of the matrix
start = timer()
x1 = np.linalg.solve(S_dense, b)
end = timer()
print(f'Time it takes conventional solver:{end - start}')


# Use the solver using sparse structure and of course, the sparse form.
start = timer()
x2 = spsolve(S_10000, b)
end = timer()
print(f'Time it takes conventional solver using sparse matrices:{end - start}')



# Just produce the solution using "inverse of A times b".
start = timer()
x3 = np.linalg.inv(S_dense)@b
end = timer()
print(f'Time it takes to compute the inverse of A:{end - start}')


# Try the Jacobi iteration.
#Jacobi has the form x^k+1 = (I-D^-1A)x^k + D^-1 b 
B_Jacobi = np.eye (10000) - (-1/4) * S_dense
start = timer()
x4 = np.ones(10000)
#Performing 100 iterations:
for j in range(100):
    x4 = B_Jacobi @ x4 + (-1/4) * b
end = timer()
print(f"The exact error of our Jacobi iterations is {np.linalg.norm(x4-x_known)}" )
print(f'Time it takes for the Jacobi iteration:{end - start}')


# Try the previous one but deal the matrix as a sparse one.
from scipy.sparse import eye
B_Jacobi = eye (10000) - (-1/4) * S_10000
# Here be careful
B_Jacobi = csr_matrix(B_Jacobi)
start = timer()
x4 = np.ones(10000)
for j in range(100):
    x4 = B_Jacobi @ x4 + (-1/4) * b
end = timer()
print(f"The exact error of our Jacobi iterations is {np.linalg.norm(x4-x_known)}" )

print(f'Time it takes for the Jacobi iteration and sparse matrices:{end - start}')


Time it takes conventional solver:2.0456470000208355
Time it takes conventional solver using sparse matrices:0.001878124981885776
Time it takes to compute the inverse of A:7.293593834008789
The exact error of our Jacobi iterations is 4.665754777514033e-15
Time it takes for the Jacobi iteration:1.2069020419730805
The exact error of our Jacobi iterations is 4.665754777514033e-15
Time it takes for the Jacobi iteration and sparse matrices:0.0017567919858265668
